# Experiment 2

### Second experiment to see how class and trip purpose together affect satisfaction

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

### Imports

In [9]:
import joblib
import pandas as pd

from src.config import MODEL_DIR, RATING_COLS
from src.data import load_train, get_split
from src.preprocessing import apply_preprocessing

### Load Data

In [3]:
X, y = load_train()
X_tr, X_val, y_tr, y_val = get_split(X, y)

saved = joblib.load(MODEL_DIR / "logreg_onehot_ratings.joblib")
state, model = saved["state"], saved["model"]

### Errors

In [4]:
val = X_val.copy()
val["actual"] = y_val
val["proba"] = model.predict_proba(apply_preprocessing(X_val, state))[:, 1]
val["wrong"] = (val["proba"] >= 0.5).astype(int) != val["actual"]

print("Overall error rate:", val["wrong"].mean().round(4))

Overall error rate: 0.0915


In [5]:
val.groupby(["Type of Travel", "Class"])["wrong"].agg(["mean", "size"]).round(3)

mean   size
Type of Travel  Class                 
Business travel Business  0.091  67629
                Eco       0.112  28060
                Eco Plus  0.129   3518
Personal Travel Business  0.123    816
                Eco       0.073  37378
                Eco Plus  0.073   2526

In [6]:
val.groupby("Customer Type")["wrong"].agg(["mean", "size"]).round(3)

,mean,size
Customer Type,,
Loyal Customer,0.089,115440
disloyal Customer,0.101,24487


- The errors are spread evenly, between 7% and 13% across every Type of Travel × Class group, and the delay, loyalty and unrated-service cuts show only small gaps (8.2% vs 9.3%, 8.9% vs 10.1%, 9.0% vs about 11%). 
- The groups with the highest rates are tiny, so even fixing them would remove about 100 mistakes out of 12.8k. That’s why we dropped the Type of Travel × Class interaction and the delay-feature experiments, as there was no evidence they would help.

In [10]:
val["any_delay"] = (val["Departure Delay in Minutes"] > 0) | (val["Arrival Delay in Minutes"] > 0)
val["n_unrated"] = (val[RATING_COLS] == 0).sum(axis=1)

print(val.groupby("any_delay")["wrong"].agg(["mean", "size"]).round(3))
print(val.groupby("n_unrated")["wrong"].agg(["mean", "size"]).round(3))

            mean    size
any_delay               
False      0.093  121338
True       0.082   18589
            mean    size
n_unrated               
0          0.090  130871
1          0.116    6020
2          0.110    1030
3          0.081    1422
4          0.080     574
5          0.000      10


In [11]:
wrong = val[val["wrong"]]
print("Mistakes with probability between 0.3 and 0.7:",
      wrong["proba"].between(0.3, 0.7).mean().round(3))

Mistakes with probability between 0.3 and 0.7: 0.346


- Only 34.6% of the wrong predictions had a probability between 0.3 and 0.7. The other roughly 8,400 were clearly wrong. A model that is only slightly off tends to miss by a little, so mostly confident misses point to missing structure. Logistic regression gives each column one fixed weight, so it can’t capture combinations such as “a poor seat only matters when legroom is also poor”.